# 02 — Fall datasets → skeleton shards (ONLINE, GPU, internet ON)

Same extraction recipe as notebook 01, over the fall corpora: **Le2i, CAUCAFall,
GMDCSA-24, URFD**. A few GB total — one session.

| attach as input | produces |
|---|---|
| `behaviorsense-code` | `behaviorsense-fall-shards` |
| your uploaded copies of Le2i / CAUCAFall / URFD (mirror fallback) | |

Academic fall-dataset mirrors are famously flaky, so each source is fetched in a
try/except and the honest fallback is attaching a copy you uploaded as a private dataset.
The `datasets` column records the source per window — that column is what makes the P2
leave-one-dataset-out protocol possible later, so it is not optional metadata.

In [ ]:
import subprocess, sys
# onnxruntime-gpu 1.27+ is built against CUDA 13; Kaggle ships CUDA 12, so the CUDA
# provider fails to load (libcublasLt.so.13 missing) and RTMO silently runs on CPU.
# 1.26.x is the newest CUDA-12.8 build - the same toolkit torch 2.7.1+cu128 uses.
# --no-deps on rtmlib: it depends on the CPU `onnxruntime`, which would overwrite the
# GPU package's provider registration.
subprocess.run(["pip", "uninstall", "-y", "onnxruntime"], check=False)
subprocess.run(["pip", "install", "-q", "--no-deps", "rtmlib"], check=True)
# av: PyAV links its OWN ffmpeg, which is why it can read clips that abort cv2's.
subprocess.run(["pip", "install", "-q", "onnxruntime-gpu==1.26.0", "pyyaml", "tqdm",
                "av>=12.0"], check=True)

In [ ]:
# Resolve the repo mount by CONTENT, not by dataset name.
#
# The dataset title is free text and this project has already been uploaded under more
# than one spelling ("behaviorsense-*" and "behavioursense-*"). Hard-coding the name makes
# cell 1 of a 12-hour session fail on a typo, so find the repo by a file only it contains.
import pathlib

INPUT = pathlib.Path("/kaggle/input")
def attached_mounts():
    # Datasets do NOT sit directly under /kaggle/input. They mount at
    # /kaggle/input/datasets/<owner>/<name>/, and competitions at
    # /kaggle/input/competitions/<name>/. Listing INPUT.iterdir() therefore always
    # reports ['competitions', 'datasets'] whatever is attached - which is what the
    # "code: nothing matches ..." failure printed, telling us nothing about whether
    # the code dataset was attached. Descend to the level that names real mounts, and
    # report what each one CONTAINS, since a dataset can be attached and still be
    # missing the directory the notebook needs.
    out = []
    for container in ("datasets", "competitions"):
        base = INPUT / container
        if not base.is_dir():
            continue
        for owner in sorted(base.iterdir()):
            kids = sorted(owner.iterdir()) if owner.is_dir() else []
            if kids and all(k.is_dir() for k in kids[:1]) and container == "datasets":
                for ds in kids:
                    top = sorted(q.name for q in ds.iterdir())[:6] if ds.is_dir() else []
                    out.append(f"{ds.name} (top level: {top})")
            else:
                out.append(owner.name)
    # Fall back to the flat layout so this keeps working if Kaggle changes the mount
    # shape back, rather than reporting nothing at all.
    return out or sorted(p.name for p in INPUT.iterdir())

ATTACHED = attached_mounts() if INPUT.is_dir() else []
_init = sorted(INPUT.glob("**/src/behaviorsense/__init__.py"))
assert _init, (f"repo dataset not found: nothing matches **/src/behaviorsense/__init__.py "
               f"under /kaggle/input. Attached datasets: {ATTACHED}")
SRC     = _init[0].parent.parent
CODE    = SRC.parent
SCRIPTS = CODE / "scripts"
CONFIGS = CODE / "configs"
import sys
sys.path.insert(0, str(SRC)); sys.path.insert(0, str(SCRIPTS))
print(f"  code {CODE}")
print(f"  attached {ATTACHED}")

CONTRACT = [
    ("scripts/kaggle_smoke_test.py", "--profile",      "notebook 03 preflight"),
    ("scripts/train_adl.py",         "--stop-after",   "resume guard (notebook 03)"),
    ("scripts/train_fall.py", "pos_rate > 0.5 and args.focal_alpha > 0.5",
     "refuses focal alpha that up-weights the majority; a snapshot without it trains "
     "the fall head on 82% positives and reports a plausible but meaningless AUPRC"),
    ("scripts/prepare_skeletons.py", "def assign_slots",
     "slot tracking + windowing (notebooks 01/02)"),
    ("scripts/prepare_skeletons.py", "with_starts",
     "fall labelling by true frame position (notebook 02); index-derived position "
     "mislabels the descent whenever a window is dropped"),
    ("scripts/prepare_skeletons.py", "def le2i_fall_frames",
     "Le2i has no fall/ADL marker in any path component; without this its ~192 fall "
     "clips land in the negatives (notebook 02)"),
    ("scripts/prepare_skeletons.py", "def label_fall_windows",
     "shared fall labelling: exact interval for Le2i, positional fallback elsewhere"),
    ("src/behaviorsense/data/skeleton_dataset.py", "keep_root_motion",
     "falls are unlearnable without it"),
    ("src/behaviorsense/models/ensemble.py", "def per_stream_logits", "notebook 04 ablation"),
    ("src/behaviorsense/models/stgcnpp.py", "parent.setdefault",
     "flip-equivariant bone stream; without it half the ensemble trains on sign noise"),
    ("src/behaviorsense/kaggle_artifacts.py", "def find_run_dir",
     "one rule for 'is this real session output or a dev leftover'; four call sites "
     "learned it separately and the fourth was missed"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def repair_claim",
     "format-only claim repair + maxItems bound to max_claims (notebook 04). Without "
     "it the unconstrained arm scores 245 emitted / 0 scorable / nan%, which measures "
     "JSON compliance rather than faithfulness"),
    ("scripts/eval_hallucination.py", "unusable_rate",
     "three-arm hallucination table with a denominator over EMITTED claims; a stale "
     "snapshot silently reports the two-arm nan% version"),
    ("src/behaviorsense/pipeline.py", "def frames_to_windows", "Agent 1 -> Agent 2 seam"),
    ("configs/taxonomy.yaml",        None,             "class map (notebook 01)"),
]
_stale = []
for _rel, _token, _why in CONTRACT:
    _p = CODE / _rel
    if not _p.is_file():
        _stale.append(f"{_rel} is MISSING ({_why})")
        continue
    if _token and _token not in _p.read_text(encoding="utf-8", errors="ignore"):
        _stale.append(f"{_rel} lacks {_token!r} ({_why})")
if _stale:
    raise AssertionError(
        "The attached code dataset is OLDER than these notebooks:\n  - "
        + "\n  - ".join(_stale)
        + f"\n\nMounted: {CODE}\nRe-upload from your checkout, then restart this notebook:"
          "\n  kaggle datasets version -p . --dir-mode zip -m \"sync\""
    )
print(f"  contract {len(CONTRACT)}/{len(CONTRACT)} - mounted code is current")

In [ ]:
from rtmlib import RTMO
# Same staged rtmo-l.onnx as notebook 01. This notebook has internet and rtmlib could
# fetch its own copy, but then the ADL and fall shards would come from two separately
# resolved checkpoints - the fall head would be trained on a different pose distribution
# than the ADL heads, and nothing would say so.
_onnx = sorted(pathlib.Path("/kaggle/input").glob("**/rtmo-l.onnx"))
assert _onnx, ("rtmo-l.onnx not found - attach the weights dataset from notebook 00 "
               f"(e.g. behavioursense-WW). Mounted: {sorted(p.name for p in pathlib.Path('/kaggle/input').iterdir())}")
body = RTMO(onnx_model=str(_onnx[0]), model_input_size=(640, 640),
            backend="onnxruntime", device="cuda")
# Same live-provider check as notebook 01: get_available_providers() reports the build,
# the session reports reality. A CUDA-13 onnxruntime-gpu on a CUDA-12 image lists the
# provider, fails to load it, and extracts on CPU without saying so.
_active = list(getattr(body, "session").get_providers())
assert "CUDAExecutionProvider" in _active, (
    f"RTMO is running on {_active} - CUDA did not load. This notebook has internet, so "
    "fix it here: pip install onnxruntime-gpu==1.26.0 (newest CUDA 12.8 build).")
print(f"pose model: {_onnx[0]}  provider: {_active[0]}")

In [ ]:
# Acquire all four fall corpora. Three fetch themselves; Le2i attaches as a mount.
#
# Verified reachable 2026-08-08. Le2i is the exception: its canonical host
# (le2i.cnrs.fr) refuses connections, the IMVIA successor page 404s, and the Wayback
# captures under that domain are staff pages with no archived video - so it is consumed
# as an attached Kaggle dataset instead of a download. Attach `tuyenldvn/falldataset-imvia`
# ("Le2i Fall Dataset", ~10 GB); the mount is found by CONTENT below, so any mirror or
# any dataset title works.
#
# Each corpus is independent: one unreachable source must not cost the others a session,
# so failures are collected and reported, never raised.
import subprocess, pathlib, urllib.request, urllib.error, json, zipfile, shutil
import sys as _sys
_sys.path.insert(0, str(SCRIPTS))
from prepare_skeletons import le2i_fall_frames

TMP = pathlib.Path("/tmp/falls"); TMP.mkdir(parents=True, exist_ok=True)
INPUT = pathlib.Path("/kaggle/input")
SOURCES, FAILED = {}, {}

# Mendeley answered 403 to urllib's default User-Agent from a Kaggle IP (it works from
# a laptop), so every request here carries a browser UA. Same opener is used for the
# API walk below, or the walk 403s while the file fetches succeed.
_UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}
_opener = urllib.request.build_opener()
_opener.addheaders = list(_UA.items())
urllib.request.install_opener(_opener)

def _get(url, dst, timeout=120):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 1000:
        return True
    try:
        req = urllib.request.Request(url, headers=_UA)
        with urllib.request.urlopen(req, timeout=timeout) as r, open(dst, "wb") as f:
            shutil.copyfileobj(r, f)
        return dst.stat().st_size > 1000
    except (urllib.error.URLError, OSError, TimeoutError):
        dst.unlink(missing_ok=True)
        return False

# --- GMDCSA-24: git clone -------------------------------------------------------------
try:
    d = TMP / "gmdcsa"
    if not d.exists():
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/ekramalam/GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos",
            str(d)], check=True, capture_output=True)
    SOURCES["gmdcsa"] = d
except Exception as exc:
    FAILED["gmdcsa"] = f"clone failed: {exc}"

# --- URFD: 70 direct MP4s. The host every paper cites (fenix.univ.rzeszow.pl) is dead;
# fenix.ur.edu.pl is the live one. Its primary distribution is zipped PNG frames, but we
# decode video anyway, so the MP4s are the cheaper path.
try:
    d = TMP / "urfd"; base = "https://fenix.ur.edu.pl/~mkepski/ds/data"
    want = ([(f"fall-{i:02d}-cam0.mp4") for i in range(1, 31)]
            + [f"adl-{i:02d}-cam0.mp4" for i in range(1, 41)])
    got = sum(_get(f"{base}/{n}", d / n) for n in want)
    if got:
        SOURCES["urfd"] = d
    if got < len(want):
        print(f"  urfd      {got}/{len(want)} clips (partial - continuing)")
    if not got:
        FAILED["urfd"] = "no clips downloaded"
except Exception as exc:
    FAILED["urfd"] = str(exc)

# --- CAUCAFall: ATTACHED MOUNT, not a download. Its Mendeley record (DOI
# 10.17632/7w7fccy7ky) holds nine documentation files - one xlsx and eight jpegs, 1.1 MB
# total - and no video at all; versions 1-3 answer 451 and the S3 bulk zip is 403 under any
# User-Agent. So a walk of that record can only ever find figures, which is exactly what
# "Mendeley walk found no video" meant. Attach `tuyenldvn/caucafall` (8.33 GB) instead -
# same publisher as the Le2i mirror already in use.
try:
    cauca = None
    for cand in INPUT.glob("**/*"):
        # Subject.1 .. Subject.10, each holding one folder per activity. The corpus root is
        # their parent, found by content so the mirror's nesting depth does not matter.
        if not cand.is_dir():
            continue
        if cand.name.lower().replace(" ", "").replace("_", "") not in ("subject.1", "subject1"):
            continue
        if any(cand.parent.rglob("*.avi")) or any(cand.parent.rglob("*.mp4")):
            cauca = cand.parent
            break
    if cauca is not None:
        SOURCES["caucafall"] = cauca
    else:
        FAILED["caucafall"] = ("not attached - add the Kaggle dataset tuyenldvn/caucafall "
                               "(8.33 GB). Mendeley publishes only figures for this DOI.")
except Exception as exc:
    FAILED["caucafall"] = str(exc)

# --- Le2i: attached mount, located by CONTENT. Its annotation .txt files are the only
# thing that distinguishes its fall clips from its ADL clips, so the mount is identified
# by having BOTH video and Annotation_files - a video-only mirror is refused loudly in
# the labelling cell rather than silently labelled ADL.
# The root must be the CORPUS root, not the mount. Kaggle nests datasets as
# /kaggle/input/datasets/<owner>/<name>/..., and iterating INPUT.glob("*") accepted
# `/kaggle/input/datasets` itself - so every Le2i clip's subject id became the OWNER
# name, collapsing all 190 videos onto one id and defeating the P1 split. Anchor on the
# annotations instead: Annotation_files sits at <corpus>/<scene>/Annotation_files, so
# its grandparent is the corpus root and the scene survives as the subject.
le2i = None
_anno_dirs = sorted(INPUT.glob("**/Annotation_files"))
if _anno_dirs:
    # The corpus root is the COMMON ancestor of every annotation folder, not the
    # grandparent of the first one. Taking the grandparent and breaking picked up
    # Coffee_room_01 alone - 48 of ~190 videos, and since subject ids are derived
    # relative to the root, all 48 collapsed onto the single id "le2i_Coffee_room_01".
    # commonpath spans the scenes whatever depth the mirror nests them at, and keeps the
    # scene as the first relative component, which is what becomes the subject.
    import os
    if len(_anno_dirs) == 1:
        _root = _anno_dirs[0].parent.parent
    else:
        _root = pathlib.Path(os.path.commonpath([str(a) for a in _anno_dirs]))
    _cands = [_root]
else:   # some mirrors drop the folder and keep the .txt beside the video
    _cands = sorted({t.parent.parent for t in INPUT.glob("**/*.txt")
                     if (t.parent / f"{t.stem}.avi").exists()
                     or (t.parent.parent / "Videos" / f"{t.stem}.avi").exists()})
for cand in _cands:
    vids = sorted(cand.rglob("*.avi"))
    if not vids:
        continue
    # Functional check, not a shape check: annotations must actually RESOLVE. Sample
    # ACROSS the tree rather than the first 20 - a contiguous head sample lives in one
    # scene and says nothing about the rest.
    step = max(1, len(vids) // 20)
    sample = vids[::step][:20]
    hits = sum(le2i_fall_frames(v) is not None for v in sample)
    if hits >= max(1, len(sample) // 2):
        le2i = cand
        scenes = sorted({v.relative_to(cand).parts[0] for v in vids})
        print(f"  le2i      annotations resolve for {hits}/{len(sample)} sampled clips "
              f"across {len(scenes)} scene(s): {scenes[:6]}")
        break
if le2i:
    SOURCES["le2i"] = le2i
else:
    FAILED["le2i"] = (f"no attached mount has resolvable Le2i annotations "
                      f"(checked {len(_cands)} candidate root(s)) - attach "
                      "tuyenldvn/falldataset-imvia")

for k in ("gmdcsa", "urfd", "caucafall", "le2i"):
    if k in SOURCES:
        n = sum(1 for _ in SOURCES[k].rglob("*") if _.suffix.lower() in (".avi", ".mp4"))
        print(f"  {k:<10} OK    {n:>4} videos  {SOURCES[k]}")
    else:
        print(f"  {k:<10} ABSENT      {FAILED.get(k, '?')}")
assert SOURCES, f"no corpus acquired: {FAILED}"
print(f"\n{len(SOURCES)}/4 corpora - P2 leave-one-dataset-out needs >=2 to mean anything")

In [ ]:
# Decode probe: find the clips that KILL the interpreter, before spending a session.
#
# Two runs died here, both inside URFD, both ~0.4 s after three "[mp3float] Header
# missing" lines. Neither the per-video try/except nor the 3000-frame cap fired, because
# the fault is not in Python: it is in cv2 -> ffmpeg -> the mp3float decoder, and a
# SIGSEGV/abort there takes the whole interpreter with it. No in-process guard can catch
# that. (Checked: all 70 URFD mp4s are well-formed video-only mp42 containers, none
# truncated - so this is not a bad download and cannot be screened by inspection.)
#
# So decode every clip ONCE in a child process first, journalling each attempt BEFORE it
# starts. If the child dies, its last unfinished entry names the poison file; the parent
# restarts and the child skips everything already journalled. Converges in 1 + n_poison
# runs, each crash costing exactly one clip instead of the corpus.
#
# This costs a decode pass (~5-8 min) and buys a session that finishes.
import subprocess, sys, pathlib

WORK = pathlib.Path("/kaggle/working")
JOURNAL = WORK / "decode_probe.tsv"
LISTFILE = WORK / "decode_list.txt"

all_vids = []
for _src, _root in SOURCES.items():
    all_vids += sorted(str(v) for v in _root.rglob("*")
                       if v.suffix.lower() in (".mp4", ".avi"))
LISTFILE.write_text(chr(10).join(all_vids) + chr(10), encoding="utf-8")
print(f"probing {len(all_vids)} clips for decoder crashes")

TAB, NL = chr(9), chr(10)
CHILD = NL.join([
    "import sys, os",
    "backend = sys.argv[3] if len(sys.argv) > 3 else 'cv2'",
    "lst, out = sys.argv[1], sys.argv[2]",
    "seen = set()",
    "if os.path.exists(out):",
    "    for line in open(out):",
    "        parts = line.rstrip(chr(10)).split(chr(9))",
    "        if len(parts) >= 2:",
    "            seen.add(parts[1])",
    "f = open(out, 'a', buffering=1)",
    "for line in open(lst):",
    "    p = line.strip()",
    "    if not p or p in seen:",
    "        continue",
    "    f.write('TRY' + chr(9) + p + chr(10))",   # journalled BEFORE the risky decode
    "    n = 0",
    "    if backend == 'av':",
    "        import av",
    "        with av.open(p) as container:",
    "            for _fr in container.decode(video=0):",
    "                n += 1",
    "                if n > 20000:",
    "                    break",
    "    else:",
    "        import cv2",
    "        cap = cv2.VideoCapture(p)",
    "        while True:",
    "            ok, _fr = cap.read()",
    "            if not ok:",
    "                break",
    "            n += 1",
    "            if n > 20000:",
    "                break",
    "        cap.release()",
    "    f.write('OK' + chr(9) + p + chr(9) + str(n) + chr(10))",
])
(WORK / "_decode_probe.py").write_text(CHILD, encoding="utf-8")

# One restart per poison clip. A cap of 11 was set for "a couple of bad files" and was
# immediately wrong: Le2i's mirror crashes on EVERY clip, so the budget ran out after 11
# and the remaining 179 were never probed - silently excluded from extraction while the
# log said only "continuing with what passed". Each restart costs ~0.25 s, so budgeting
# for the worst case (every clip poison) costs ~2 min and removes the failure mode.
# Progress is guaranteed: the child journals TRY before decoding, so each run advances at
# least one clip.
budget = len(all_vids) + 5
for attempt in range(1, budget + 1):
    r = subprocess.run([sys.executable, str(WORK / "_decode_probe.py"),
                        str(LISTFILE), str(JOURNAL), "cv2"],
                       capture_output=True, text=True)
    if r.returncode == 0:
        break
    if attempt <= 3 or attempt % 25 == 0:
        print(f"  probe crashed (exit {r.returncode}) - restart {attempt}/{budget}")
else:
    print(f"  WARNING: probe never converged in {budget} restarts")

# Second chance with PyAV before writing anything off. cv2 SIGABRTs/SIGSEGVs on every
# clip in the Le2i mirror - 190 videos, including the only corpus here with exact
# frame-level fall annotations - and that is a codec the bundled ffmpeg mishandles, not
# corrupt data. PyAV links its own ffmpeg, so it is a genuinely different decoder, and
# `av>=12.0` is already declared in requirements-train.txt for exactly this job.
#
# This is self-verifying: if PyAV also dies, those clips stay excluded and the only cost
# is a few minutes of probing. Nothing is assumed to work.
_rows = [l.split(TAB) for l in JOURNAL.read_text(encoding="utf-8").splitlines() if l.strip()]
_ok_cv2 = {r[1] for r in _rows if r[0] == "OK"}
_failed = sorted(set(all_vids) - _ok_cv2)
AV_JOURNAL = WORK / "decode_probe_av.tsv"
AV_OK = set()
if _failed:
    print(f"  retrying {len(_failed)} cv2-failed clip(s) with PyAV")
    AVLIST = WORK / "decode_list_av.txt"
    AVLIST.write_text(chr(10).join(_failed) + chr(10), encoding="utf-8")
    for attempt in range(1, len(_failed) + 5):
        r = subprocess.run([sys.executable, str(WORK / "_decode_probe.py"),
                            str(AVLIST), str(AV_JOURNAL), "av"],
                           capture_output=True, text=True)
        if r.returncode == 0:
            break
    if AV_JOURNAL.exists():
        AV_OK = {q.split(TAB)[1] for q in
                 AV_JOURNAL.read_text(encoding="utf-8").splitlines()
                 if q.startswith("OK")}
    print(f"  PyAV rescued {len(AV_OK)}/{len(_failed)}")

rows = [l.split(TAB) for l in JOURNAL.read_text(encoding="utf-8").splitlines() if l.strip()]
USABLE = {r[1] for r in rows if r[0] == "OK"} | AV_OK
# Per-clip backend, so extraction decodes each file with whatever actually read it.
BACKEND = {q: ("av" if q in AV_OK else "cv2") for q in USABLE}
POISON = sorted(set(all_vids) - USABLE)
UNPROBED = sorted(set(all_vids) - USABLE - set(POISON))
print(f"  {len(USABLE)}/{len(all_vids)} clips decode cleanly")
if POISON:
    # Named, not silently dropped: real data is being excluded and the count belongs in
    # the write-up next to the shard totals.
    by_dir = {}
    for q in POISON:
        by_dir[str(pathlib.Path(q).parent)] = by_dir.get(str(pathlib.Path(q).parent), 0) + 1
    print(f"  {len(POISON)} clip(s) crash the decoder, by directory:")
    for d, n in sorted(by_dir.items(), key=lambda kv: -kv[1]):
        print(f"    {n:>4}  {d}")
if UNPROBED:
    # Distinct from POISON: these were never even attempted, so nothing is known about
    # them. Reporting them as one number with the crashes would hide a budget failure.
    print(f"  {len(UNPROBED)} clip(s) NEVER PROBED (budget exhausted) - also excluded")
assert USABLE, "no clip decoded cleanly - do NOT Save Version"


In [ ]:
# Extract + label. Fall clips are short and pre-trimmed; windows are labelled by clip
# type and position: the descent lands mid-clip, so windows around the drop are `falling`
# (7), later windows `fallen_on_ground` (8), earlier ones `standing` (1); ADL clips ->
# other_idle (19). Coarse by construction — the OmniFall staged->wild protocol is where
# finer temporal labels come from; this labelling is stated in the eval, not hidden.
import cv2, collections, pathlib
import numpy as np
from prepare_skeletons import (   # tested repo code, not copies
    WINDOW_FRAMES, assign_slots, is_fall_clip, label_fall_windows, le2i_fall_frames,
    subject_from_path, window_clip)
OUT = pathlib.Path("/kaggle/working/shards"); OUT.mkdir(exist_ok=True)

def extract_av(video, fps_sample=15):
    # PyAV path, for the clips whose codec aborts cv2. Frames come out RGB; RTMO was fed
    # BGR for every other corpus, so convert - mixing colour orders across corpora would
    # be a silent distribution shift in the fall head's training data.
    import av
    poses, i, step = [], 0, 1
    with av.open(str(video)) as container:
        stream = container.streams.video[0]
        src = float(stream.average_rate or 25)
        step = max(1, round(src / fps_sample))
        for frame in container.decode(video=0):
            if i % step == 0:
                kp, sc = body(frame.to_ndarray(format="bgr24"))
                poses.append(assign_slots(kp, sc, poses[-1] if poses else None))
                if len(poses) >= 3000:
                    break
            i += 1
    return (np.stack(poses) if len(poses) >= 30 else None), step


def extract(video, fps_sample=15):
    # -> (poses, step). step is returned because Le2i's annotations are in ORIGINAL
    # frame numbers while the windows are in subsampled ones; the caller needs it to
    # convert. A docstring cannot go here - this cell is itself a '''...''' literal.
    if BACKEND.get(str(video)) == "av":
        return extract_av(video, fps_sample)
    cap = cv2.VideoCapture(str(video))
    src = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(1, round(src / fps_sample))
    # No clip in these corpora runs past a few minutes, so an unbounded read loop buys
    # nothing and risks everything: a malformed container hands back frame after frame
    # from a broken index until RAM is gone. The run that died logged three
    # "[mp3float] Header missing" lines and lost the kernel on the next one.
    MAX_SAMPLED = 3000                      # 200 s at 15 fps
    poses, f, capped = [], 0, False
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if f % step == 0:
            kp, sc = body(frame)
            # Tracked slots, not per-frame area order - see notebook 01's extraction cell.
            poses.append(assign_slots(kp, sc, poses[-1] if poses else None))
            if len(poses) >= MAX_SAMPLED:
                capped = True
                break
        f += 1
    cap.release()
    if capped:
        print(f"    CAPPED {video.name} at {MAX_SAMPLED} sampled frames")
    return (np.stack(poses) if len(poses) >= 30 else None), step

skels, labels, subjects, sources = [], [], [], []
unlabelled = collections.Counter()
shard_i, banked = 0, 0

# Flush every ~400 MB and at every corpus boundary, exactly like notebook 01. The first
# version accumulated all four corpora and wrote ONE npz after the loop; the kernel died
# 27 minutes in, on a Le2i clip, and took the already-finished gmdcsa (160 clips) and
# urfd (70 clips) with it. Nothing about that extraction was wrong - the run simply had
# no way to keep what it had already earned.
def flush():
    global skels, labels, subjects, sources, shard_i, banked
    if not skels:
        return
    np.savez_compressed(
        OUT / f"falls_{shard_i:04d}.npz",
        skeletons=np.stack(skels).astype(np.float16),
        labels=np.asarray(labels, dtype=np.int64),
        subjects=np.asarray(subjects, dtype="<U32"),
        datasets=np.asarray(sources, dtype="<U32"))
    banked += len(skels)
    print(f"    wrote falls_{shard_i:04d}.npz ({len(skels)} windows, {banked} banked)")
    shard_i += 1
    skels, labels, subjects, sources = [], [], [], []
for source, root in SOURCES.items():
    if not root.exists():
        continue
    vids = sorted(v for v in list(root.rglob("*.mp4")) + list(root.rglob("*.avi"))
                  if str(v) in USABLE)
    print(f"{source}: {len(vids)} clips (probe-cleared)")
    for v in vids:
        # One unreadable clip must not end the corpus, let alone the session.
        try:
            poses, step = extract(v)
        except Exception as exc:                  # noqa: BLE001
            unlabelled[f"{source}:read-error"] += 1
            print(f"    SKIP {v.name}: {type(exc).__name__} {exc}")
            continue
        if poses is None:
            continue
        # How a clip is labelled depends on what its corpus actually tells us.
        #
        # Le2i is the awkward one: no path component says fall or ADL - every clip is
        # `video (N).avi` under a room folder, and the truth lives in an Annotation_files
        # .txt whose first two lines are the fall's start/end frame. Treating a missing
        # annotation as "not a fall" would write the ~192 clips that DO contain falls
        # into the negatives, so an unlabellable clip is SKIPPED and counted, never
        # guessed. Where the annotation exists it beats the heuristic outright: real
        # frame boundaries instead of "the descent is somewhere in the middle".
        fall_range = None
        if source == "le2i":
            ann = le2i_fall_frames(v)
            if ann is None:
                unlabelled[source] += 1
                continue
            fall_range = ann
            is_fall = ann != (0, 0)
        else:
            # Directory/filename markers, token-matched: CAUCAFall names its folders
            # "Fall forward"/"Fall backward", which an exact-match rule scored as ADL -
            # all 50 of its fall clips would have become negatives.
            is_fall = is_fall_clip(root, v)

        wins = window_clip(poses, with_starts=True)
        if not wins:
            continue
        starts = [st for st, _ in wins]
        if is_fall or fall_range is not None:
            # Sampled space: annotations are in ORIGINAL frames, windows are in
            # subsampled frames, so the interval is converted with the same step the
            # decoder used. Skipping this scales the fall interval by ~2x at 15 fps.
            rng = None if fall_range is None else (fall_range[0] // step,
                                                   fall_range[1] // step)
            labs = label_fall_windows(starts, WINDOW_FRAMES, rng)
        else:
            labs = [19] * len(starts)

        subj = subject_from_path(root, v, source)
        for (_st, w), lab in zip(wins, labs):
            skels.append(w)
            labels.append(lab)
            subjects.append(subj)
            sources.append(source)

        if sum(x.nbytes for x in skels) > 400e6:
            flush()
    # Corpus boundary: bank what is finished before starting the next one.
    flush()
    # Report what this corpus actually CONTRIBUTED, not just the running total. The run
    # that motivated this acquired 190 Le2i clips and shipped 3 of its 6 scenes; the
    # difference was counted in `unlabelled` and never printed, so the loss was invisible
    # in a log that otherwise looked healthy.
    _used = sum(1 for q in vids if str(q) in USABLE)
    _skipped = sum(n for k, n in unlabelled.items() if k.startswith(source))
    _scenes = sorted({subject_from_path(root, q, source) for q in vids
                      if str(q) in USABLE}) if _used else []
    print(f"  {source} done - {banked} windows on disk | {_used}/{len(vids)} clips "
          f"decodable, {_skipped} skipped, {len(_scenes)} subject id(s) contributed")
    if _skipped:
        for k, n in sorted(unlabelled.items()):
            if k.startswith(source):
                print(f"      skipped: {k.split(':', 1)[-1] if ':' in k else 'no-annotation'}"
                      f" x{n}")

# Gate before writing. Notebook 01 taught this the hard way: a silent zero-yield run that
# still invites "Save Version" publishes an unusable dataset as if it were fine, and Kaggle
# versions REPLACE content. Every assert below is a thing that has a plausible silent path.
# Read back from DISK, not from the in-memory lists: extraction now flushes every 400 MB
# and at every corpus boundary, so by the time this cell runs those lists are empty by
# design. Checking them would assert "NO windows extracted" on a perfectly good run - and
# a gate that cries wolf is a gate that gets commented out. Only the label/subject/dataset
# vectors are loaded; the skeletons stay on disk.
if unlabelled:
    print("\nclips skipped during extraction (counted, and now reported):")
    for k, n in sorted(unlabelled.items()):
        print(f"  {k:<34} {n:>5}")
    print("  A clip is skipped when its label cannot be established - for Le2i that means")
    print("  no readable Annotation_files entry. Skipping is correct (guessing would put")
    print("  real falls in the negatives), but the COUNT belongs in the write-up.")

shards = sorted(OUT.glob("falls_*.npz"))
assert shards, (
    f"NO shard written from {list(SOURCES)}. Nothing to save - do NOT Save Version. "
    f"Check the clone succeeded and that clips are .mp4/.avi under those roots.")
labels, subjects, sources = [], [], []
for sp in shards:
    with np.load(sp, allow_pickle=False) as z:
        labels.extend(z["labels"].tolist())
        subjects.extend(z["subjects"].tolist())
        sources.extend(z["datasets"].tolist())
print(f"{len(shards)} shard(s) on disk, {len(labels)} windows total")

counts = collections.Counter(labels)
names = {1: "standing", 7: "falling", 8: "fallen_on_ground", 19: "other_idle"}
print(f"{len(labels)} windows from {len(set(sources))} source(s):")
for lab in sorted(counts):
    print(f"  {lab:>2} {names.get(lab, '?'):<18} {counts[lab]:>6}")

# The fall head is a BINARY discriminator. Without both fall and non-fall windows it
# trains on one class, reports a meaningless AUROC near 0.5, and nothing in the metric
# says the data was degenerate rather than the model bad.
have_fall = counts[7] + counts[8]
have_neg  = counts[1] + counts[19]
assert have_fall > 0, (
    f"ZERO fall windows ({dict(counts)}). The is_fall rule matched no clip - it keys on "
    f"'fall' in the filename or parent directory, so a corpus using other names (e.g. "
    f"'Coup'/'Chute') needs the rule extended. Do NOT Save Version: notebook 03's fall "
    f"head would train on a single class.")
assert have_neg > 0, (
    f"ZERO non-fall windows ({dict(counts)}) - every clip matched the is_fall rule, so "
    f"the binary head has no negatives. Check the ADL clips are actually present.")
assert counts[7] > 0, (
    f"fall clips found but ZERO 'falling' windows ({dict(counts)}) - the descent labelling "
    f"failed. Expected at least one per fall clip.")
print(f"  balance: {have_fall} fall / {have_neg} non-fall")

# Subject ids decide the P1 split, and a collapsed id is INVISIBLE downstream:
# split_by_subject asserts the ids are disjoint, which they are - it is the PEOPLE behind
# them that would not be. GMDCSA-24 numbers every subject's clips 01..25, so the old
# stem-based rule mapped four people onto one id and put each of them on both sides of
# the split. Print the ids and refuse a degenerate count.
by_subject = collections.Counter(subjects)
print(f"  {len(by_subject)} subject id(s):")
for sub, n in sorted(by_subject.items()):
    print(f"    {sub:<28} {n:>6} windows")
assert len(by_subject) >= 2, (
    f"only {len(by_subject)} subject id ({list(by_subject)}) - cross-subject (P1) "
    "evaluation needs at least 2, and split_by_subject would raise on a degenerate "
    "split. Check subject_from_path against this corpus's directory layout.")
# Every subject should carry both kinds of clip, or the split can hand the fall head a
# validation fold containing no falls at all.
sub_has_fall = collections.defaultdict(set)
for sub, lab in zip(subjects, labels):
    sub_has_fall[sub].add(lab in (7, 8))
one_sided = [k for k, v in sub_has_fall.items() if len(v) == 1]
if one_sided:
    print(f"  NOTE: {len(one_sided)} subject(s) have only one clip type "
          f"({one_sided[:4]}) - a split isolating them yields a fold with no positives.")

# Subject-id GRANULARITY, stated because it decides what P1 actually measures.
#
# URFD is flat (fall-01-cam0.mp4) and publishes no clip->volunteer mapping - only
# per-frame posture labels - so subject_from_path falls back to the filename stem and
# every clip becomes its own "subject". Its ~70 sequences come from a handful of
# volunteers, so cross-subject splitting cannot separate them and P1 is OPTIMISTIC for
# the URFD portion. That is a property of the corpus, not a bug to code around, and it
# belongs in the eval tables next to the number - exactly as the Charades video-id proxy
# already is.
#
# The measurable consequence is variance: holding out 20% of SUBJECTS holds out anywhere
# from 3% to 45% of WINDOWS when most ids are single clips. Measured on this shard set,
# val folds ranged 160-1406 windows across 12 seeds. So report the spread rather than
# trusting one draw.
tiny = [k for k, n in by_subject.items() if n < 20]
print(f"\n  subject granularity: {len(by_subject)} ids, {len(tiny)} with <20 windows")
if tiny:
    per_ds = collections.Counter(k.split("_")[0] for k in tiny)
    print(f"    small ids by corpus: {dict(per_ds)}")
    print("    URFD publishes no clip->volunteer mapping, so each CLIP is its own subject.")
    print("    P1 is therefore optimistic for URFD - state this in the eval table.")
from behaviorsense.data.skeleton_dataset import split_by_subject as _sbs
_subs = np.asarray(subjects)
_sizes = []
for _seed in range(8):
    try:
        _tr, _va = _sbs(_subs, val_frac=0.2, seed=_seed)
        _sizes.append(len(_va))
    except Exception:
        _sizes.append(0)
print(f"    val-fold size across 8 seeds: min {min(_sizes)} / max {max(_sizes)} windows")
if min(_sizes) < 200:
    print("    WARNING: at least one seed yields a val fold under 200 windows. Fit the")
    print("    fall operating point on a fixed seed and report the fold size with it.")

# P2 is leave-one-DATASET-out (docs/06 section 'protocols'). One corpus cannot support it:
# there is no second dataset to hold out. This is a loud warning, not an assert, because a
# single-corpus shard is still fine for P1/P3 and for a first end-to-end pass.
n_src = len(set(sources))
if n_src < 2:
    print(f"\n  WARNING: only {n_src} dataset ({sorted(set(sources))}). P2 "
          f"leave-one-dataset-out is NOT possible - it needs at least 2 corpora, "
          f"ideally 3-4.")
    print("           Notebook 04 will skip the P2 table and say so. To enable it, upload "
          "Le2i / CAUCAFall / URFD as private datasets and uncomment them in SOURCES above.")
else:
    print(f"\n  P2 viable: {n_src} datasets -> {n_src} leave-one-out folds "
          f"({sorted(set(sources))})")

# Nothing to write here: extraction already banked every window. A savez at this point
# would rebuild falls_0000.npz from the now-empty in-memory lists and OVERWRITE the
# real first shard with an empty one - the flush refactor's sharpest edge.
mb = sum(sp.stat().st_size for sp in shards) / 1e6
print(f"\n{len(shards)} shard(s), {mb:.1f} MB total:")
for sp in shards:
    print(f"  {sp.name}  {sp.stat().st_size/1e6:>7.1f} MB")
print("\nSave Version -> create dataset behaviorsense-fall-shards.")